In [1]:
# 配置路径并列出千烟洲30分钟气象数据文件
from pathlib import Path
import re
import os
from datetime import datetime
import pandas as pd
import numpy as np
import re
base_dir = Path("气象数据/30分钟")
assert base_dir.exists(), f"目录不存在: {base_dir}"

files = sorted([p.name for p in base_dir.iterdir() if p.is_file()])
for f in files:
    print(f)

2011年千烟洲气象30分钟数据.xls
2012年千烟洲气象30分钟数据.xls
2013年千烟洲气象30分钟数据.xls
2014年千烟洲气象30分钟数据.xls
2015年千烟洲气象30分钟数据.xls


In [2]:
# 将2011-2015的文件名重命名为：{年}年千烟洲气象30分钟数据.{原扩展名}
pattern = re.compile(r"^(20\d{2})年千烟洲站森林综合观测场气象30分钟数据-?0825\.(xls|xlsx)$")

rename_actions = []
for p in base_dir.iterdir():
    if not p.is_file():
        continue
    m = pattern.match(p.name)
    if not m:
        continue
    year, ext = m.group(1), m.group(2)
    new_name = f"{year}年千烟洲气象30分钟数据.{ext}"
    src = p
    dst = p.with_name(new_name)
    if src.name == dst.name:
        continue
    if dst.exists():
        continue
    os.rename(src, dst)
    rename_actions.append((src.name, dst.name))
for s, d in rename_actions:
    print(f"  {s}  =>  {d}")

In [3]:
BASE_DIR = Path('气象数据/30分钟')
years = list(range(2011, 2016))
def find_year_file(y: int) -> Path | None:
    for ext in ('.xlsx', '.xls'):
        p = BASE_DIR / f'{y}年千烟洲气象30分钟数据{ext}'
        if p.exists() and p.is_file():
            return p
    for p in BASE_DIR.glob(f'{y}年千烟洲气象30分钟数据.*'):
        if p.is_file():
            return p
    return None

files = [find_year_file(y) for y in years]
files = [p for p in files if p is not None]

print('待合并文件(2011-2015):')
for p in files:
    print(' -', p.name)
if len(files) == 0:
    raise FileNotFoundError('未找到 2011-2015 年的已重命名文件。')

frames = []
reference_cols = None
reference_src = None

for fp in files:
    engine = 'xlrd' if fp.suffix.lower() == '.xls' else 'openpyxl'
    raw = pd.read_excel(fp, header=None, engine=engine)
    if raw.empty:
        continue
    header = raw.iloc[0].astype(str).str.strip().tolist()
    df = raw.iloc[2:].copy()  # 跳过第1行表头 + 第2行单位
    df.columns = header
    df.reset_index(drop=True, inplace=True)

    df.columns = [str(c).strip().replace(' ', '') for c in df.columns]

    if reference_cols is None:
        reference_cols = list(df.columns)
        reference_src = fp.name

    year = None
    for y in years:
        if str(y) in fp.name:
            year = y
            break
    df.insert(0, '来源年份', year)

    frames.append(df.reindex(columns=['来源年份'] + reference_cols))

merged = pd.concat(frames, ignore_index=True)
print('合并完成:', len(merged), '行; 文件数:', len(frames))
print('基准表头来自:', reference_src)

待合并文件(2011-2015):
 - 2011年千烟洲气象30分钟数据.xls
 - 2012年千烟洲气象30分钟数据.xls
 - 2013年千烟洲气象30分钟数据.xls
 - 2014年千烟洲气象30分钟数据.xls
 - 2015年千烟洲气象30分钟数据.xls
合并完成: 87648 行; 文件数: 5
基准表头来自: 2011年千烟洲气象30分钟数据.xls


In [4]:
# 2) 将11-15年的通量30分钟数据合并成一张表（忽略第二行单位，不导出）
import pandas as pd
from pathlib import Path

FLUX_DIR = Path('通量数据/30分钟')
assert FLUX_DIR.exists(), f'目录不存在: {FLUX_DIR}'

years = list(range(2011, 2016))

def find_year_file(y: int) -> Path | None:
    # 先尝试常见扩展名
    for ext in ('.xlsx', '.xls'):
        for p in FLUX_DIR.glob(f'*{y}*通量*{ext}'):
            if p.is_file():
                return p
    # 兜底：任何扩展名，只要文件名包含年份与“通量”
    for p in FLUX_DIR.glob(f'*{y}*通量*'):
        if p.is_file():
            return p
    return None

files = [find_year_file(y) for y in years]
files = [p for p in files if p is not None]

print('待合并通量文件(2011-2015):')
for p in files:
    print(' -', p.name)
if len(files) == 0:
    raise FileNotFoundError('未找到 2011-2015 年的通量文件。')

frames = []
reference_cols = None
reference_src = None

for fp in files:
    # 第一行是列名，第二行是单位 -> 跳过
    engine = 'xlrd' if fp.suffix.lower() == '.xls' else 'openpyxl'
    raw = pd.read_excel(fp, header=None, engine=engine)
    if raw.empty:
        continue
    header = raw.iloc[0].astype(str).str.strip().tolist()
    df = raw.iloc[2:].copy()  # 跳过第1行表头 + 第2行单位
    df.columns = header
    df.reset_index(drop=True, inplace=True)

    df.columns = [str(c).strip().replace(' ', '') for c in df.columns]

    if reference_cols is None:
        reference_cols = list(df.columns)
        reference_src = fp.name

    year = None
    for y in years:
        if str(y) in fp.name:
            year = y
            break
    df.insert(0, '来源年份', year)

    # 对齐列顺序，缺失列补NaN，多余列丢弃
    frames.append(df.reindex(columns=['来源年份'] + reference_cols))

merged_flux_1115 = pd.concat(frames, ignore_index=True)

待合并通量文件(2011-2015):
 - 2011年千烟洲站森林综合观测场通量30分钟数据 -0825.xls
 - 2012年千烟洲站森林综合观测场通量30分钟数据 - 0825.xls
 - 2013年千烟洲站森林综合观测场通量30分钟数据 - 0825.xls
 - 2014年千烟洲站森林综合观测场通量30分钟数据 - 0825.xls
 - 2015年千烟洲站森林综合观测场通量30分钟数据 - 0825.xls


In [5]:
def _find_col(df: pd.DataFrame, keys):
    for col in df.columns:
        low = str(col).lower()
        if any(k in low for k in keys):
            return col
    return None


def build_datetime(df: pd.DataFrame) -> pd.Series:

    one = _find_col(df, ['时间','日期','时刻','timestamp','datetime','datet'])
    if one is not None:
        dt = pd.to_datetime(df[one], errors='coerce')
        if dt.notna().any():
            return dt
    y = _find_col(df, ['年','year'])
    m = _find_col(df, ['月','month'])
    d = _find_col(df, ['日','day'])
    h = _find_col(df, ['时','小时','hour'])
    mi = _find_col(df, ['分','分钟','min'])
    se = _find_col(df, ['秒','sec','second'])

    if y and m and d:
        ys = pd.to_numeric(df[y], errors='coerce').astype('Int64')
        ms = pd.to_numeric(df[m], errors='coerce').fillna(1).astype('Int64')
        ds = pd.to_numeric(df[d], errors='coerce').fillna(1).astype('Int64')
        hs = pd.to_numeric(df[h], errors='coerce').fillna(0).astype('Int64') if h else 0
        mis = pd.to_numeric(df[mi], errors='coerce').fillna(0).astype('Int64') if mi else 0
        ss = pd.to_numeric(df[se], errors='coerce').fillna(0).astype('Int64') if se else 0
        return pd.to_datetime({'year': ys, 'month': ms, 'day': ds, 'hour': hs, 'minute': mis, 'second': ss}, errors='coerce')

    for col in df.columns:
        dt = pd.to_datetime(df[col], errors='coerce')
        if dt.notna().any():
            return dt

    return pd.Series(pd.NaT, index=df.index)

# 复制数据以避免污染原DataFrame
wx = merged.copy() if ('merged' in globals() and isinstance(merged, pd.DataFrame)) else None
fx = merged_flux_1115.copy() if ('merged_flux_1115' in globals() and isinstance(merged_flux_1115, pd.DataFrame)) else None

if wx is None or fx is None:
    raise RuntimeError('请先运行：第4格(气象2011-2015合并)与第6格(通量2011-2015合并)。')

# 构造 datetime
wx_dt = build_datetime(wx)
fx_dt = build_datetime(fx)
wx.insert(0, 'datetime', wx_dt)
fx.insert(0, 'datetime', fx_dt)
wx = wx[wx['datetime'].notna()].copy()
fx = fx[fx['datetime'].notna()].copy()

# 排序以便直观检查
wx.sort_values('datetime', inplace=True)
fx.sort_values('datetime', inplace=True)

print('气象：', len(wx), '行; 时间范围:', wx['datetime'].min(), '->', wx['datetime'].max())
print('通量：', len(fx), '行; 时间范围:', fx['datetime'].min(), '->', fx['datetime'].max())

# 处理同名列冲突：给通量侧除 datetime 外的列添加后缀
wx_cols = set(wx.columns)
fx_renamed = fx.rename(columns={c: (c + '_通量') if c != 'datetime' and c in wx_cols else c for c in fx.columns})

# 内连接对齐
wx_flux_aligned = pd.merge(wx, fx_renamed, on='datetime', how='inner')
wx_flux_aligned.sort_values('datetime', inplace=True)
wx_flux_aligned.reset_index(drop=True, inplace=True)

print('对齐完成：', len(wx_flux_aligned), '行')
print('对齐时间范围：', wx_flux_aligned['datetime'].min(), '->', wx_flux_aligned['datetime'].max())

气象： 87648 行; 时间范围: 2011-01-01 00:00:00 -> 2015-12-31 23:30:00
通量： 87648 行; 时间范围: 2011-01-01 00:00:00 -> 2015-12-31 23:30:00
对齐完成： 87648 行
对齐时间范围： 2011-01-01 00:00:00 -> 2015-12-31 23:30:00


In [6]:
# 4) 筛选保留的字段并清除无关指标（在时间对齐后）
import pandas as pd

if 'wx_flux_aligned' not in globals() or not isinstance(wx_flux_aligned, pd.DataFrame):
    raise RuntimeError('请先运行时间对齐单元格以生成 wx_flux_aligned。')

# 指定要排除的字段（来源年份、日期分解、通量侧重复的日期分解、部分通量指标）
columns_to_exclude = {
    '来源年份', '年', '月', '日', '时', '分',
    '年_通量', '月_通量', '日_通量', '时_通量', '分_通量',
    'RE', 'GEE', 'LE', 'Hs'
}
# 动态排除可能出现的通量侧来源年份
if '来源年份_通量' in wx_flux_aligned.columns:
    columns_to_exclude.add('来源年份_通量')

# 计算保留列（保留 datetime 与其余未排除的列，保持原始顺序）
keep_cols = ['datetime'] + [
    c for c in wx_flux_aligned.columns
    if c != 'datetime' and c not in columns_to_exclude
]

wx_flux_selected = wx_flux_aligned.loc[:, keep_cols].copy()

print('筛选后列数:', len(wx_flux_selected.columns))
print('筛选后行数:', len(wx_flux_selected))
print('保留列示例:', keep_cols[:15])

# # 预览
# display(wx_flux_selected.head(5))
# display(wx_flux_selected.tail(5))

筛选后列数: 22
筛选后行数: 87648
保留列示例: ['datetime', '近地面空气温度', '冠层上方空气温度', '近地面空气相对湿度', '冠层上方空气相对湿度', '近地面水汽压', '冠层上方水汽压', '近地面风速', '冠层上方风速', '风向', '大气压', '太阳辐射', '净辐射', '光合有效辐射', '一层土壤温度']


In [7]:
# 5) 基于一层/二层/三层合成“土壤温度”和“土壤含水量”，并替换原6列
import pandas as pd

if 'wx_flux_selected' not in globals() or not isinstance(wx_flux_selected, pd.DataFrame):
    raise RuntimeError('请先运行字段筛选单元格，确保存在 wx_flux_selected。')

df = wx_flux_selected.copy()

temp_cols = ['一层土壤温度','二层土壤温度','三层土壤温度']
vwc_cols  = ['一层土壤体积含水量','二层土壤体积含水量','三层土壤体积含水量']

# 仅使用存在的列，避免个别年份缺列导致报错
temp_present = [c for c in temp_cols if c in df.columns]
vwc_present  = [c for c in vwc_cols  if c in df.columns]

if len(temp_present) == 0 or len(vwc_present) == 0:
    raise RuntimeError(f'找不到必要列: 温度{temp_present}, 含水{vwc_present}。请检查前序步骤的列名。')

# 合成：对三层做行均值（忽略NaN）
df['土壤温度']  = df[temp_present].mean(axis=1, skipna=True)
df['土壤含水量'] = df[vwc_present].mean(axis=1, skipna=True)

# 删除原6列
df.drop(columns=temp_present + vwc_present, inplace=True)

# 重新排布列：将新列放到datetime之后
ordered_cols = ['datetime','土壤温度','土壤含水量'] + [c for c in df.columns if c not in ['datetime','土壤温度','土壤含水量']]
df = df.loc[:, ordered_cols]

wx_flux_selected = df  # 覆盖为更新后的结果

print('替换完成：')
print('列数:', len(wx_flux_selected.columns), '行数:', len(wx_flux_selected))
print('首列示例:', wx_flux_selected.columns[:12].tolist())

# display(wx_flux_selected.head(5))
# display(wx_flux_selected.tail(5))

替换完成：
列数: 18 行数: 87648
首列示例: ['datetime', '土壤温度', '土壤含水量', '近地面空气温度', '冠层上方空气温度', '近地面空气相对湿度', '冠层上方空气相对湿度', '近地面水汽压', '冠层上方水汽压', '近地面风速', '冠层上方风速', '风向']


In [8]:
# 5.1) 最终列筛选与导出：保留用户指定列并导出CSV
import pandas as pd
from pathlib import Path

if 'wx_flux_selected' not in globals() or not isinstance(wx_flux_selected, pd.DataFrame):
    raise RuntimeError('请先运行前序单元格，确保存在 wx_flux_selected。')

df = wx_flux_selected.copy()

# 若通量侧出现重名导致NEE被标为NEE_通量，则统一为NEE
if 'NEE_通量' in df.columns and 'NEE' not in df.columns:
    df.rename(columns={'NEE_通量': 'NEE'}, inplace=True)

required_cols = [
    'datetime',
    '土壤温度', '土壤含水量',
    '净辐射', '降水量', '光合有效辐射', '太阳辐射', '大气压', '风向',
    '冠层上方风速', '近地面风速',
    '冠层上方水汽压', '近地面水汽压',
    '冠层上方空气相对湿度', '近地面空气相对湿度',
    '冠层上方空气温度', '近地面空气温度',
    'NEE'
]

present = [c for c in required_cols if c in df.columns]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    print('提示：以下指定列在当前数据中未找到 ->', missing)

df = df.loc[:, present]
wx_flux_selected = df

print('最终列筛选完成：', len(wx_flux_selected.columns), '列; ', len(wx_flux_selected), '行')
print('列清单：', wx_flux_selected.columns.tolist())

# # 导出CSV
# output_dir = Path('分析输出')
# output_dir.mkdir(parents=True, exist_ok=True)
# final_csv = output_dir / '千烟洲_2011-2015_气象通量_精选列.csv'
# wx_flux_selected.to_csv(final_csv, index=False, encoding='utf-8-sig')
# print('已导出最终精选列数据到：', final_csv)

最终列筛选完成： 18 列;  87648 行
列清单： ['datetime', '土壤温度', '土壤含水量', '净辐射', '降水量', '光合有效辐射', '太阳辐射', '大气压', '风向', '冠层上方风速', '近地面风速', '冠层上方水汽压', '近地面水汽压', '冠层上方空气相对湿度', '近地面空气相对湿度', '冠层上方空气温度', '近地面空气温度', 'NEE']


In [9]:
# 6) 缺失值分析：统计各列缺失数与缺失率，并按缺失率排序
import pandas as pd

if 'wx_flux_selected' not in globals() or not isinstance(wx_flux_selected, pd.DataFrame):
    raise RuntimeError('请先运行前序筛选与合成单元格，确保存在 wx_flux_selected。')

df = wx_flux_selected.copy()

total_rows = len(df)
missing_cnt = df.isna().sum()
nonmissing_cnt = df.notna().sum()
dtypes = df.dtypes.astype(str)

stats = pd.DataFrame({
    '缺失数': missing_cnt,
    '非缺失数': nonmissing_cnt,
    '缺失率(%)': (missing_cnt / total_rows * 100).round(2),
    '数据类型': dtypes,
})
stats.index.name = '列名'

stats_sorted = stats.sort_values('缺失率(%)', ascending=False)

print('总行数:', total_rows)
print('有缺失的列数:', int((missing_cnt > 0).sum()))
print('完全无缺失的列数:', int((missing_cnt == 0).sum()))

# 展示缺失率最高的前20列
print('\n缺失率Top20列:')
display(stats_sorted.head(20))

# 如需完整表，可取消下一行注释
# display(stats_sorted)


总行数: 87648
有缺失的列数: 3
完全无缺失的列数: 15

缺失率Top20列:


,缺失数,非缺失数,缺失率(%),数据类型
列名,,,,
净辐射,8,87640,0.01,object
datetime,0,87648,0.00,datetime64[ns]
近地面风速,0,87648,0.00,object
近地面空气温度,0,87648,0.00,object
冠层上方空气温度,0,87648,0.00,object
近地面空气相对湿度,0,87648,0.00,object
冠层上方空气相对湿度,0,87648,0.00,object
近地面水汽压,0,87648,0.00,object
冠层上方水汽压,0,87648,0.00,object


In [11]:
# 7.1) 全列数值化 + 相关性（Pearson/Spearman）并导出
import pandas as pd
import numpy as np
from pathlib import Path

if 'wx_flux_selected' not in globals() or not isinstance(wx_flux_selected, pd.DataFrame):
    raise RuntimeError('请先生成 wx_flux_selected（运行前序合并、对齐与筛选单元）。')

# 复制并尝试将除 datetime 外的所有列数值化
all_df = wx_flux_selected.copy()
for c in all_df.columns:
    if c == 'datetime':
        continue
    # 对 object 列做基础清洗后转为数值
    if all_df[c].dtype == 'object':
        s = all_df[c].astype(str)
        s = (
            s.str.replace(',', '', regex=False)
             .str.replace(' ', '', regex=False)
             .str.replace('\u3000', '', regex=False)
             .str.replace('\t', '', regex=False)
             .str.replace('—', '', regex=False)
             .str.replace('−', '-', regex=False)
        )
        all_df[c] = pd.to_numeric(s, errors='coerce')
    else:
        all_df[c] = pd.to_numeric(all_df[c], errors='coerce')

num_df_all = all_df.select_dtypes(include=['number']).copy()
print('数值化后用于相关性分析的列数:', num_df_all.shape[1], '行数:', num_df_all.shape[0])

# 计算 Pearson 与 Spearman 相关系数
corr_pearson = num_df_all.corr(method='pearson')
corr_spearman = num_df_all.corr(method='spearman')
print('Pearson矩阵维度:', corr_pearson.shape)
print('Spearman矩阵维度:', corr_spearman.shape)

# # 导出到 CSV
# output_dir = Path('分析输出')
# output_dir.mkdir(parents=True, exist_ok=True)

# out_p = output_dir / '千烟洲_2011-2015_气象通量_相关性矩阵_全列_Pearson.csv'
# out_s = output_dir / '千烟洲_2011-2015_气象通量_相关性矩阵_全列_Spearman.csv'

# corr_pearson.to_csv(out_p, encoding='utf-8-sig')
# corr_spearman.to_csv(out_s, encoding='utf-8-sig')

# print('已导出Pearson相关矩阵到:', out_p)
# print('已导出Spearman相关矩阵到:', out_s)

# 额外：输出Top相关对

def top_pairs(corr: pd.DataFrame, topn: int = 30) -> pd.DataFrame:
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = upper.stack().rename('相关系数')
    # 依据绝对值排序，但保留原符号
    pairs_abs_sorted = pairs.abs().sort_values(ascending=False)
    pairs_sorted = pairs.loc[pairs_abs_sorted.index]
    df_pairs = pairs_sorted.reset_index()
    df_pairs.columns = ['变量A', '变量B', '相关系数']
    return df_pairs.head(topn)

top30_p = top_pairs(corr_pearson, 30)
top30_s = top_pairs(corr_spearman, 30)

# out_tp = output_dir / '千烟洲_2011-2015_气象通量_相关性Top30_Pearson.csv'
# out_ts = output_dir / '千烟洲_2011-2015_气象通量_相关性Top30_Spearman.csv'

# top30_p.to_csv(out_tp, index=False, encoding='utf-8-sig')
# top30_s.to_csv(out_ts, index=False, encoding='utf-8-sig')

# print('已导出Top30相关对(Pearson)到:', out_tp)
# print('已导出Top30相关对(Spearman)到:', out_ts)

# 简要预览Top10（Pearson）
display(top30_p.head(10))

数值化后用于相关性分析的列数: 17 行数: 87648
Pearson矩阵维度: (17, 17)
Spearman矩阵维度: (17, 17)


,变量A,变量B,相关系数
0,冠层上方水汽压,近地面水汽压,0.994181
1,光合有效辐射,太阳辐射,0.992400
2,冠层上方空气温度,近地面空气温度,0.990638
3,净辐射,太阳辐射,0.987414
4,净辐射,光合有效辐射,0.982482
5,土壤温度,近地面水汽压,0.937887
6,土壤温度,冠层上方水汽压,0.931902
7,冠层上方空气相对湿度,近地面空气相对湿度,0.918352
8,土壤温度,近地面空气温度,0.915291
9,土壤温度,冠层上方空气温度,0.911806


In [ ]:
# # 8) 随机森林测试：预测目标列 NEE 并分析特征重要性
# import pandas as pd
# import numpy as np
# from pathlib import Path

# # 依赖检查
# try:
#     from sklearn.ensemble import RandomForestRegressor
#     from sklearn.model_selection import train_test_split
#     from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
#     from sklearn.inspection import permutation_importance
# except Exception as e:
#     raise RuntimeError('缺少 scikit-learn 依赖，请安装后重试。错误: ' + str(e))

# # 1) 读取数据：优先使用内存中的 wx_flux_selected；若不存在则从导出的精选列CSV加载
# if 'wx_flux_selected' in globals() and isinstance(wx_flux_selected, pd.DataFrame):
#     df = wx_flux_selected.copy()
# else:
#     csv_path = Path('分析输出') / '千烟洲_2011-2015_气象通量_精选列.csv'
#     if not csv_path.exists():
#         raise RuntimeError('未找到数据：请先运行合并/筛选并导出，或确保存在 分析输出/千烟洲_2011-2015_气象通量_精选列.csv')
#     df = pd.read_csv(csv_path)

# # 2) 基础清洗：将除 datetime 外的所有列数值化；NEE 作为目标
# # 若通量侧出现重名导致NEE被标为NEE_通量，则统一为NEE
# if 'NEE_通量' in df.columns and 'NEE' not in df.columns:
#     df.rename(columns={'NEE_通量': 'NEE'}, inplace=True)

# if 'NEE' not in df.columns:
#     raise RuntimeError('未找到目标列 NEE，请检查前序步骤的列筛选。')

# # 清洗 object 列为数值（去除逗号/空格等）
# def to_numeric_series(s: pd.Series) -> pd.Series:
#     if s.dtype == 'object':
#         s2 = s.astype(str)
#         s2 = (
#             s2.str.replace(',', '', regex=False)
#                .str.replace(' ', '', regex=False)
#                .str.replace('\u3000', '', regex=False)
#                .str.replace('\t', '', regex=False)
#                .str.replace('—', '', regex=False)
#                .str.replace('−', '-', regex=False)
#         )
#         return pd.to_numeric(s2, errors='coerce')
#     return pd.to_numeric(s, errors='coerce')

# for c in df.columns:
#     if c == 'datetime':
#         # 若 datetime 列是字符串，尝试解析为时间
#         try:
#             df[c] = pd.to_datetime(df[c], errors='coerce')
#         except Exception:
#             pass
#     else:
#         df[c] = to_numeric_series(df[c])

# # 3) 构造特征与目标：去除 datetime；保留数值特征
# feature_cols = [c for c in df.columns if c not in ['datetime', 'NEE']]
# X = df[feature_cols].copy()
# y = df['NEE'].copy()

# # 丢弃任一特征或目标为 NaN 的行
# valid_mask = X.notna().all(axis=1) & y.notna()
# X = X.loc[valid_mask]
# y = y.loc[valid_mask]

# # 若 datetime 存在用于时间排序分割
# if 'datetime' in df.columns and pd.api.types.is_datetime64_any_dtype(df['datetime']):
#     dt_valid = df.loc[valid_mask, 'datetime']
#     # 按时间排序后作时间序列分割：后20%为测试集
#     order = dt_valid.sort_values().index
#     X = X.loc[order]
#     y = y.loc[order]

# # 时间序列比例划分（不打乱）
# split_idx = int(len(X) * 0.8)
# X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
# y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# print('训练/测试规模:', len(X_train), '/', len(X_test))
# print('特征列数:', len(feature_cols))

# # 4) 训练随机森林回归模型
# rf = RandomForestRegressor(
#     n_estimators=400,
#     max_depth=None,
#     min_samples_split=2,
#     min_samples_leaf=1,
#     random_state=42,
#     n_jobs=-1,
# )
# rf.fit(X_train, y_train)

# # 5) 评估指标
# pred_train = rf.predict(X_train)
# pred_test = rf.predict(X_test)

# r2_tr = r2_score(y_train, pred_train)
# r2_te = r2_score(y_test, pred_test)
# mae_tr = mean_absolute_error(y_train, pred_train)
# mae_te = mean_absolute_error(y_test, pred_test)
# rmse_tr = mean_squared_error(y_train, pred_train, squared=False)
# rmse_te = mean_squared_error(y_test, pred_test, squared=False)

# print('R2(train/test):', round(r2_tr, 4), '/', round(r2_te, 4))
# print('MAE(train/test):', round(mae_tr, 4), '/', round(mae_te, 4))
# print('RMSE(train/test):', round(rmse_tr, 4), '/', round(rmse_te, 4))

# # 6) 特征重要性（基于不纯度）
# impurity_imp = pd.DataFrame({
#     '特征': feature_cols,
#     '不纯度重要性': rf.feature_importances_
# }).sort_values('不纯度重要性', ascending=False)

# # 7) 置换重要性（基于测试集）
# perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
# perm_imp = pd.DataFrame({
#     '特征': feature_cols,
#     '置换重要性(均值)': perm.importances_mean,
#     '置换重要性(标准差)': perm.importances_std,
# }).sort_values('置换重要性(均值)', ascending=False)

# # 8) 合并两类重要性并导出
# # out_dir = Path('分析输出')
# # out_dir.mkdir(parents=True, exist_ok=True)

# # merged_imp = pd.merge(impurity_imp, perm_imp, on='特征', how='outer')
# # imp_csv = out_dir / '千烟洲_2011-2015_NEE_随机森林_特征重要性.csv'
# # merged_imp.to_csv(imp_csv, index=False, encoding='utf-8-sig')
# # print('已导出特征重要性到:', imp_csv)

# # 9) 同步导出评估指标
# # metrics_csv = out_dir / '千烟洲_2011-2015_NEE_随机森林_评估指标.csv'
# # metrics_df = pd.DataFrame([
# #     {
# #         'R2_train': r2_tr,
# #         'R2_test': r2_te,
# #         'MAE_train': mae_tr,
# #         'MAE_test': mae_te,
# #         'RMSE_train': rmse_tr,
# #         'RMSE_test': rmse_te,
# #         '训练样本数': len(X_train),
# #         '测试样本数': len(X_test),
# #         '特征数': len(feature_cols),
# #     }
# # ])
# # metrics_df.to_csv(metrics_csv, index=False, encoding='utf-8-sig')
# # print('已导出评估指标到:', metrics_csv)

# # 10) 预览Top15特征
# print('\nTop15(不纯度重要性):')
# display(impurity_imp.head(15))
# print('\nTop15(置换重要性):')
# display(perm_imp.head(15))

In [12]:
columns_to_remove = [
    "近地面空气温度",
    "近地面空气相对湿度",
    "光合有效辐射",
    "降水量",
    "太阳辐射",
    "近地面水汽压",
    "大气压",
    "冠层上方风速",
]

# 从 wx_flux_selected 中移除上述列，并导出一个新CSV
import pandas as pd
from pathlib import Path

if 'wx_flux_selected' not in globals() or not isinstance(wx_flux_selected, pd.DataFrame):
    raise RuntimeError('请先生成 wx_flux_selected（运行字段筛选与合成单元）。')

df = wx_flux_selected.copy()
existing = [c for c in columns_to_remove if c in df.columns]
missing = [c for c in columns_to_remove if c not in df.columns]
if existing:
    df.drop(columns=existing, inplace=True)
wx_flux_selected = df

print('已移除列:', existing)
if missing:
    print('未找到且跳过的列:', missing)
print('当前列数:', len(wx_flux_selected.columns), '行数:', len(wx_flux_selected))

已移除列: ['近地面空气温度', '近地面空气相对湿度', '光合有效辐射', '降水量', '太阳辐射', '近地面水汽压', '大气压', '冠层上方风速']
当前列数: 10 行数: 87648


In [17]:
# 9) 清洗、过滤NEE科学计数法、四舍五入、重命名并导出为 QianYanZhou.csv
import pandas as pd
import numpy as np
from pathlib import Path

# 选择数据源：优先使用内存中的 wx_flux_selected，回退到已导出的精选列CSV
if 'wx_flux_selected' in globals() and isinstance(wx_flux_selected, pd.DataFrame):
    df = wx_flux_selected.copy()
else:
    fallback = Path('分析输出') / '千烟洲_2011-2015_气象通量_精选列.csv'
    if not fallback.exists():
        raise RuntimeError('未找到可用数据：请先运行前序合并/筛选并导出，或确保存在 分析输出/千烟洲_2011-2015_气象通量_精选列.csv')
    df = pd.read_csv(fallback)

# 若通量侧出现重名导致NEE被标为NEE_通量，则统一为NEE
if 'NEE' not in df.columns and 'NEE_通量' in df.columns:
    df = df.rename(columns={'NEE_通量': 'NEE'})

# 先依据原始字符串过滤：删除 NEE 中含有 'e' 或 'E'（科学计数法表示）的行
if 'NEE' in df.columns:
    nee_str = df['NEE'].astype(str)
    mask_e = nee_str.str.contains(r'[eE]', na=False)
    removed_e = int(mask_e.sum())
    if removed_e > 0:
        df = df.loc[~mask_e].copy()
else:
    removed_e = 0

# 将除 datetime 外的列尽量数值化，便于统一处理 -9999
for c in df.columns:
    if c == 'datetime':
        try:
            df[c] = pd.to_datetime(df[c], errors='coerce')
        except Exception:
            pass
    else:
        if df[c].dtype == 'object':
            s = df[c].astype(str)
            s = (
                s.str.replace(',', '', regex=False)
                 .str.replace(' ', '', regex=False)
                 .str.replace('\u3000', '', regex=False)
                 .str.replace('\t', '', regex=False)
                 .str.replace('—', '', regex=False)
                 .str.replace('−', '-', regex=False)
            )
            df[c] = pd.to_numeric(s, errors='coerce')
        else:
            df[c] = pd.to_numeric(df[c], errors='coerce')

# 将 -9999 视为缺失
df.replace(-9999, np.nan, inplace=True)

# 需要检查缺失的列（排除 datetime）
check_cols = [c for c in df.columns if c != 'datetime']

# 丢弃包含空值（含 -9999 已转为 NaN）的行
clean = df.dropna(subset=check_cols, how='any').copy()

# 数值列四舍五入：NEE 保留8位，其余保留4位
num_cols = clean.select_dtypes(include=[np.number]).columns.tolist()
if 'NEE' in num_cols:
    clean['NEE'] = clean['NEE'].round(8)
other_num_cols = [c for c in num_cols if c != 'NEE']
if other_num_cols:
    clean[other_num_cols] = clean[other_num_cols].round(4)

# 重命名列（仅重命名存在的列）
var_name_map = {
    "datetime": "date",
    "土壤温度": "Ts",
    "土壤含水量": "SWC",
    "净辐射": "Rn",
    "风向": "WD",
    "近地面风速": "WS",
    "冠层上方水汽压": "Vp",
    "冠层上方空气相对湿度": "RH",
    "冠层上方空气温度": "Ta",
    "NEE": "NEE",
}
rename_dict = {k: v for k, v in var_name_map.items() if k in clean.columns}
clean.rename(columns=rename_dict, inplace=True)

# 导出
out_dir = Path('分析输出')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'QianYanZhou.csv'
clean.to_csv(out_path, index=False, encoding='utf-8-sig')

print('清洗完成：')
print('因NEE科学计数法被剔除的行数:', removed_e)
print('清洗后:', len(clean), '行; 列数:', len(clean.columns))
print('重命名列数:', len(rename_dict))
print('已导出到:', out_path)


清洗完成：
因NEE科学计数法被剔除的行数: 17
清洗后: 87623 行; 列数: 10
重命名列数: 10
已导出到: 分析输出/QianYanZhou.csv
